### <u> Demo Ticks </u>

In [ ]:
%reload_ext autoreload 
%autoreload 2
import matplotlib.pyplot as plt
from strauss.sonification import Sonification
from strauss.sources import Objects, Events
from strauss import channels
from strauss.score import Score
import numpy as np
from strauss.generator import Synthesizer
import IPython.display as ipd
import glob
import os
import copy
from pathlib import Path
%matplotlib inline

Let's, mimic the soundfont light-curve notebook, but just use the Synth

In [ ]:
flute_sampler = Synthesizer()
flute_sampler.load_preset('pitch_mapper')
guitar_sampler = Synthesizer()

First let's add ticks to an `Event` sonification

In [ ]:
# pick a soundfont to use

#generator = guitar_sampler
generator = copy.copy(flute_sampler)

lightcurve = np.genfromtxt(Path('..', 'data', 'datasets', '55Cancri_lc.dat'))
x = lightcurve[:,0][:]
y = lightcurve[:,1][:]

notes = [["C3","D3","E3","G3","B3","C4","D4","E4","G4","B4","C5","D5","E5","G5","B5"]]
score =  Score(notes, 15)
        
maps = {'pitch':y,
        'time': x}

system = "mono"

# manually set note properties to get a suitable sound
generator.modify_preset({'note_length':0.03, # hold each note for 0.03 seconds or 30 ms - what if this was 1s?
                         'volume_envelope': {'use':'on',
                                            # A,D,R values in seconds, S sustain fraction from 0-1 that note
                                            # will 'decay' to (after time A+D)
                                            'A':0.01,    # ✏️ Time to fade in note to maximum volume, using 10 ms
                                            'D':0.06,    # ✏️ Time to fall from maximum volume to sustained level (s), irrelevant while S is 1 
                                            'S':0.,      # ✏️ fraction of maximum volume to sustain note at while held, 1 implies 100% 
                                            'R':0.07}}) # ✏️ Time to fade out once note is released, using 100 ms

# alternatively can avoid setting manually above anf just load the 'staccato' preset
# generator.load_preset('staccato')

# set 0 to 100 percentile limits so the full pitch range is used...
# setting 0 to 101 for pitch means the sonification is 1% longer than
# the time needed to trigger each note - by making this more than 100%
# we give all the notes time to ring out (setting this at 100% means
# the final note is triggered at the momement the sonification ends)
lims = {'time': ('0%','101%'),
        'pitch': ('0%','100%')}

# set up source
sources = Events(maps.keys())
sources.fromdict(maps)
sources.apply_mapping_functions(map_lims=lims)

soni = Sonification(score, sources, generator, system)
soni.render()

# Let's add ticks at day intervals. 
# This is input in 'time' or 'time_evo' input units:
# remember if you rescale the time values before 
# input you also need to rescale this increment
# e.g. if 'time' is input is in days and increment
# soni.add_ticks(1), there will be a tick for each day
# in the data. If is it's in seconds, soni.add_ticks(120) 
# will tick every 2 minutes in the data. optional
# arguments are the duration (seconds) and volume
# (tick_vol, in amplitude fraction).
soni.add_ticks(1., duration=0.04, tick_vol=0.5)

dobj = soni.notebook_display(show_waveform=0)

plt.scatter(x,y, marker='.')
plt.ylabel('Magnitude')
plt.xlabel('Time (Julian Days)')

Now an `Objects` sonification

In [ ]:
# pick a soundfont to use

generator = copy.copy(guitar_sampler)
#generator = flute_sampler

generator.modify_preset({'filter':'on'})

# or, just load the 'sustain' preset
# generator.load_preset('sustain')

# we use a 'chord' here to create more harmonic richness (stacking fifths)...
notes = [["E2", "B3"]]
score =  Score(notes, 15)

data = {'pitch':[0,1,2,3],
        'time_evo':[x]*4,
        'cutoff':[y]*4}

lims = {'time_evo': ('0%','100%'),
        'cutoff': ('0%','100%')}

# set up source
sources = Objects(data.keys())
sources.fromdict(data)
plims = {'cutoff': (0.25,0.95)}
sources.apply_mapping_functions(map_lims=lims, param_lims=plims)

soni = Sonification(score, sources, generator, system)
soni.render()

# AGain add ticks at day intervals. 
# With the continuous sound we  use a louder, shorter tick.
# lets set it to 0.01s (10 ms). Making it very short
# can also affect it's prominence, and have spectral effects 
# (e.g. for 10ms, can't contain frequencies below 1/0.01s = 100 Hz)
soni.add_ticks(1., duration=0.01, tick_vol=1)

dobj = soni.notebook_display(show_waveform=0);
plt.plot(x,y)
plt.ylabel('Magnitude')
plt.xlabel('Time (Julian Days)')

Finally, demo the multiformat saving if wanted

In [ ]:
# soni.save("../../test.wav")
# soni.save("../../test.mp3")
# soni.save("../../test.aac")

# because we pass to ffmpeg, it can do potentially weird stuff like encode to video
# soni.save("../../test.mp4")